# Handwritten Digit Recognition: End-to-End ML Pipeline

**COM763 Advanced Machine Learning — Portfolio Task 1**

A complete pipeline for classifying 8×8 grayscale handwritten digit images into classes 0–9.

## 1. Environment Setup and Reproducibility

In [1]:
# Imports and deterministic configuration
from pathlib import Path
import json, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, ConfusionMatrixDisplay
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
ARTIFACTS = Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
print("Environment ready; random state =", RANDOM_STATE)

mkdir -p failed for path /root/.config/matplotlib: [Errno 30] Read-only file system: '/root/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-q2yt05fx because there was an issue with the default path (/root/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Environment ready; random state = 42


## 2. Problem Definition and System Framing

Input: 64 intensity values representing an 8×8 handwritten image. Output: one of ten digit classes. The pipeline keeps preprocessing inside cross-validation to prevent data leakage.

## 3. Load and Audit the Dataset

In [2]:
# Built-in Optical Recognition of Handwritten Digits dataset
digits = load_digits()
X = pd.DataFrame(digits.data, columns=[f"pixel_{i}" for i in range(64)])
y = pd.Series(digits.target, name="digit")
audit = pd.DataFrame({"Measure":["Samples","Features","Classes","Missing values","Duplicate feature rows"],
                      "Value":[len(X),X.shape[1],y.nunique(),int(X.isna().sum().sum()),int(X.duplicated().sum())]})
display(audit)
display(y.value_counts().sort_index().rename("count").to_frame().T)

                  Measure  Value
0                 Samples   1797
1                Features     64
2                 Classes     10
3          Missing values      0
4  Duplicate feature rows      0
digit    0    1    2    3    4    5    6    7    8    9
count  178  182  177  183  181  182  181  179  174  180


In [3]:
# Representative images
fig, axes = plt.subplots(2,5,figsize=(10,4))
for digit, ax in enumerate(axes.ravel()):
    idx = np.where(y.to_numpy()==digit)[0][0]
    ax.imshow(digits.images[idx], cmap="gray_r", vmin=0, vmax=16); ax.set_title(f"Label: {digit}"); ax.axis("off")
plt.suptitle("Representative handwritten digit images"); plt.tight_layout()
plt.savefig(ARTIFACTS/"sample_digits.png",dpi=180,bbox_inches="tight"); plt.show()

## 4. Data Quality and Exploratory Analysis

In [4]:
# Class balance and feature audit
counts = y.value_counts().sort_index()
ax=counts.plot(kind="bar",color="#3b82f6",figsize=(8,4),title="Class distribution")
ax.set(xlabel="Digit",ylabel="Samples"); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(ARTIFACTS/"class_distribution.png",dpi=180,bbox_inches="tight"); plt.show()
print("Imbalance ratio:",round(counts.max()/counts.min(),3))
print("Constant pixels:",int((X.var()==0).sum()),"| intensity range:",(X.min().min(),X.max().max()))

Imbalance ratio: 1.052
Constant pixels: 3 | intensity range: (np.float64(0.0), np.float64(16.0))


## 5. Stratified Train–Test Split

In [5]:
# Preserve 20% as an untouched final test set
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=RANDOM_STATE)
print("Train:",X_train.shape,"Test:",X_test.shape)
display(y_train.value_counts(normalize=True).sort_index().round(3).to_frame("proportion").T)

Train: (1437, 64) Test: (360, 64)
digit           0      1      2      3      4      5      6    7      8    9
proportion  0.099  0.102  0.099  0.102  0.101  0.101  0.101  0.1  0.097  0.1


## 6. Leakage-Safe Candidate Pipelines

In [6]:
# Three model families; transforms fit within each fold
candidates={
"Logistic Regression + PCA":Pipeline([("scale",StandardScaler()),("pca",PCA(.95,random_state=RANDOM_STATE)),("model",LogisticRegression(max_iter=3000,random_state=RANDOM_STATE))]),
"RBF SVM + PCA":Pipeline([("scale",StandardScaler()),("pca",PCA(.95,random_state=RANDOM_STATE)),("model",SVC(probability=True,random_state=RANDOM_STATE))]),
"Random Forest":Pipeline([("model",RandomForestClassifier(n_estimators=300,class_weight="balanced",random_state=RANDOM_STATE,n_jobs=-1))])}
cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE)
scoring={"accuracy":"accuracy","balanced_accuracy":"balanced_accuracy","f1_macro":"f1_macro"}

## 7. Baseline Cross-Validation and Model Comparison

In [7]:
# Identical five-fold evaluation for every candidate
rows=[]
for name,pipe in candidates.items():
    s=cross_validate(pipe,X_train,y_train,cv=cv,scoring=scoring,n_jobs=-1)
    rows.append({"Model":name,"CV Accuracy Mean":s["test_accuracy"].mean(),"CV Accuracy SD":s["test_accuracy"].std(),
                 "CV Balanced Accuracy":s["test_balanced_accuracy"].mean(),"CV Macro F1":s["test_f1_macro"].mean()})
comparison=pd.DataFrame(rows).sort_values("CV Macro F1",ascending=False).reset_index(drop=True)
display(comparison.style.format({x:"{:.4f}" for x in comparison.columns if x!="Model"}))
comparison.to_csv(ARTIFACTS/"model_comparison.csv",index=False)

In [8]:
# Comparison visual
p=comparison.sort_values("CV Macro F1")
plt.figure(figsize=(8,4)); plt.barh(p["Model"],p["CV Macro F1"],color=["#94a3b8","#60a5fa","#2563eb"])
plt.xlim(.85,1); plt.xlabel("Mean five-fold macro F1"); plt.title("Candidate model comparison"); plt.tight_layout()
plt.savefig(ARTIFACTS/"model_comparison.png",dpi=180,bbox_inches="tight"); plt.show()

## 8. Hyperparameter Tuning and Iterative Improvement

In [9]:
# Grid search on training folds only
grid={"pca__n_components":[.90,.95,.99],"model__C":[2,5,10],"model__gamma":["scale",.001,.01]}
search=GridSearchCV(candidates["RBF SVM + PCA"],grid,scoring="f1_macro",cv=cv,n_jobs=-1,return_train_score=True,refit=True)
search.fit(X_train,y_train)
print("Best parameters:",search.best_params_); print("Best CV macro F1:",round(search.best_score_,4))
cv_results=pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
display(cv_results[["params","mean_train_score","mean_test_score","std_test_score","rank_test_score"]].head(10))
cv_results.to_csv(ARTIFACTS/"grid_search_results.csv",index=False)

Best parameters: {'model__C': 2, 'model__gamma': 'scale', 'pca__n_components': 0.99}
Best CV macro F1: 0.9833
                                               params  ...  rank_test_score
2   {'model__C': 2, 'model__gamma': 'scale', 'pca_...  ...                1
18  {'model__C': 10, 'model__gamma': 'scale', 'pca...  ...                2
9   {'model__C': 5, 'model__gamma': 'scale', 'pca_...  ...                3
8   {'model__C': 2, 'model__gamma': 0.01, 'pca__n_...  ...                4
0   {'model__C': 2, 'model__gamma': 'scale', 'pca_...  ...                5
1   {'model__C': 2, 'model__gamma': 'scale', 'pca_...  ...                6
7   {'model__C': 2, 'model__gamma': 0.01, 'pca__n_...  ...                7
6   {'model__C': 2, 'model__gamma': 0.01, 'pca__n_...  ...                8
19  {'model__C': 10, 'model__gamma': 'scale', 'pca...  ...                9
10  {'model__C': 5, 'model__gamma': 'scale', 'pca_...  ...                9

[10 rows x 5 columns]


## 9. Final Evaluation on the Untouched Test Set

In [10]:
# Final evaluation occurs once after selection
best_model=search.best_estimator_; y_pred=best_model.predict(X_test)
metrics={"accuracy":accuracy_score(y_test,y_pred),"balanced_accuracy":balanced_accuracy_score(y_test,y_pred)}
report=pd.DataFrame(classification_report(y_test,y_pred,output_dict=True)).T
print({k:round(v,4) for k,v in metrics.items()}); display(report.round(4))
report.to_csv(ARTIFACTS/"classification_report.csv")

{'accuracy': 0.9778, 'balanced_accuracy': 0.9775}
              precision  recall  f1-score   support
0                1.0000  1.0000    1.0000   36.0000
1                0.9459  0.9722    0.9589   36.0000
2                1.0000  1.0000    1.0000   35.0000
3                1.0000  1.0000    1.0000   37.0000
4                0.9459  0.9722    0.9589   36.0000
5                0.9737  1.0000    0.9867   37.0000
6                0.9730  1.0000    0.9863   36.0000
7                0.9459  0.9722    0.9589   36.0000
8                1.0000  0.9143    0.9552   35.0000
9                1.0000  0.9444    0.9714   36.0000
accuracy         0.9778  0.9778    0.9778    0.9778
macro avg        0.9784  0.9775    0.9776  360.0000
weighted avg     0.9784  0.9778    0.9777  360.0000


In [11]:
# Confusion matrix
fig,ax=plt.subplots(figsize=(8,7)); ConfusionMatrixDisplay.from_predictions(y_test,y_pred,cmap="Blues",ax=ax,colorbar=False)
ax.set_title("Final model confusion matrix"); plt.tight_layout()
plt.savefig(ARTIFACTS/"confusion_matrix.png",dpi=180,bbox_inches="tight"); plt.show()

## 10. Error Analysis

In [12]:
# Inspect confident and ambiguous errors
prob=best_model.predict_proba(X_test); errors=np.flatnonzero(y_pred!=y_test.to_numpy())
print("Misclassifications:",len(errors),"of",len(y_test))
if len(errors):
    fig,axes=plt.subplots(3,4,figsize=(9,7))
    for ax in axes.ravel(): ax.axis("off")
    for ax,pos in zip(axes.ravel(),errors[:12]):
        ax.imshow(X_test.iloc[pos].to_numpy().reshape(8,8),cmap="gray_r",vmin=0,vmax=16)
        ax.set_title(f"True {y_test.iloc[pos]} | Pred {y_pred[pos]}\nConf {prob[pos].max():.2f}"); ax.axis("off")
    plt.suptitle("Representative classification errors"); plt.tight_layout()
    plt.savefig(ARTIFACTS/"error_analysis.png",dpi=180,bbox_inches="tight"); plt.show()

Misclassifications: 8 of 360


## 11. Learning Curve

In [13]:
# Diagnose bias, variance, and the value of more data
sizes,tr,val=learning_curve(best_model,X_train,y_train,cv=cv,scoring="f1_macro",train_sizes=np.linspace(.2,1,5),n_jobs=-1)
plt.figure(figsize=(8,4)); plt.plot(sizes,tr.mean(1),"o-",label="Training macro F1"); plt.plot(sizes,val.mean(1),"o-",label="Validation macro F1")
plt.fill_between(sizes,val.mean(1)-val.std(1),val.mean(1)+val.std(1),alpha=.2)
plt.xlabel("Training examples"); plt.ylabel("Macro F1"); plt.title("Learning curve"); plt.legend(); plt.tight_layout()
plt.savefig(ARTIFACTS/"learning_curve.png",dpi=180,bbox_inches="tight"); plt.show()

## 12. Export the Complete Production Pipeline

In [14]:
# Export fitted preprocessing and classifier together
joblib.dump(best_model,"digit_recognition_pipeline.joblib")
metadata={"dataset":"scikit-learn Optical Recognition of Handwritten Digits","input_shape":[8,8],"classes":list(range(10)),
          "test_accuracy":float(metrics["accuracy"]),"test_balanced_accuracy":float(metrics["balanced_accuracy"]),
          "cv_macro_f1":float(search.best_score_),"best_params":search.best_params_,"random_state":RANDOM_STATE}
Path("model_metadata.json").write_text(json.dumps(metadata,indent=2)); print(json.dumps(metadata,indent=2))

{
  "dataset": "scikit-learn Optical Recognition of Handwritten Digits",
  "input_shape": [
    8,
    8
  ],
  "classes": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9
  ],
  "test_accuracy": 0.9777777777777777,
  "test_balanced_accuracy": 0.9775396825396825,
  "cv_macro_f1": 0.983305173488812,
  "best_params": {
    "model__C": 2,
    "model__gamma": "scale",
    "pca__n_components": 0.99
  },
  "random_state": 42
}


## 13. Deployment Smoke Test

In [15]:
# Confirm that serialisation preserves predictions
reloaded=joblib.load("digit_recognition_pipeline.joblib")
assert np.array_equal(reloaded.predict(X_test.iloc[:20]),best_model.predict(X_test.iloc[:20]))
print("Smoke test passed: reloaded pipeline reproduces predictions.")

Smoke test passed: reloaded pipeline reproduces predictions.


## 14. Technical Conclusions and Limitations

Use the executed results as evidence when writing your own report. Investigate the small low-resolution dataset, performance on phone photographs or unfamiliar handwriting, confidence calibration, and appropriate educational rather than high-stakes use.